In [ ]:
import numpy as np
from netgen.csg import *
from netgen.meshing import MeshingStep
from ngsolve import *
from ngsolve.webgui import Draw
ngsglobals.msg_level = 1

# Generate mesh and plot it
center = Pnt(0,0,0)
R = 1.0
geo = CSGeometry()
geo.Add(Sphere(center, R))
maxh = 1.0
mesh = Mesh(geo.GenerateMesh(maxh=maxh, perfstepsend=MeshingStep.MESHSURFACE))
#   We use a superparametric approximation to ensure 
#   optimal convergence of solution also in w.r.t to normal component 
order_k = 3
order_g = order_k+1
mesh.Curve(order_g)
Draw(mesh)

# Check convergence of normal vector, mean curvature  (and B itself)
Hhb_errs = []
H_errs = []

def compute_eoc(errs):
    return np.log(np.array(errs)[:-1]/np.array(errs)[1:])/np.log(2)

In [ ]:
# Define geometry related objects and differential operators

# Mesh size
h = specialcf.mesh_size

# Normal vector
ns = specialcf.normal(mesh.dim)
# normal and tangential  projection
Qs = OuterProduct(ns, ns)
Ps = Id(mesh.dim) - Qs
# Weingarten map
Bs = Grad(ns)
# TODO: Or should we rather use
# specialcf.Weingarten(mesh.dim)

# Define covariant derivative for vectors
def Nabla_h(u):
    return Ps*Grad(u).Trace()*Ps
# and its corrected version for non-tangential vectors
def Nabla_th(u):
    return Nabla(u) - InnerProduct(u, ns)*Bs

# Define discrete strain rate tensor
# TODO: Need to clarify meaning/implementation of Grad(u).Trace()
#       for VectorH1
# TODO: Which projection Ps do we actually need
def E_h(u):
    return 0.5*Ps*Sym(Grad(u).Trace())*Ps
    # return 0.5*Ps*(Grad(u).Trace())
    # return Ps*(grad(u).Trace())

# Corrected strain rate tensor for non-tangential 
def E_th(u):
    return E_h(u) - InnerProduct(u, ns)*Bs

In [ ]:
V = VectorH1(mesh, order=int(order_k))
Q = H1(mesh, order=int(order_k-1))
N = NumberSpace(mesh)
W = V*Q*N

# V = HDivSurface(mesh, order=int(order_k))
# V = H1(mesh, order=int(order_k))
(u, p, lam), (v, q, mu) = W.TnT()

# Penalty parameter to weakly enforce tangential condition
beta = 1.0

Astar = BilinearForm(W)
Astar += 2*InnerProduct(E_th(u), E_th(v))*ds
# TODO: DO we need Traces here?
Astar += InnerProduct(Ps*u.Trace(), Ps*v.Trace())*ds
# TODO: Last term can replaced with
# Astar += InnerProduct(u.Trace(), v.Trace())*ds
# Penalization of normal component
# TODO:More efficient way would be to compute (u,n)*(v*n) directly
# Test that!
Astar += beta*h**(-2.0)*InnerProduct(Qs*u.Trace(), Qs*v.Trace())*ds
# Add b_h contributions 
Astar += InnerProduct(grad(p).Trace(), v.Trace())*ds
Astar += InnerProduct(grad(q).Trace(), u.Trace())*ds
Astar += (p*mu + q*lam)*ds
Astar.Assemble()

# Tangential test field
uex = CF((-y*(1-z)*(1+z), x*(1-z)*(1+z), 0))
# pex = sin(x)*sin(y)*sin(z)
pex = x*y+y*z+z*x

uex_h, pex_h, _ = GridFunction(W).components
uex_h.Set(uex, definedon=mesh.Boundaries(".*"))
pex_h.Set(pex, definedon=mesh.Boundaries(".*"))

l = LinearForm(W)
l += 2*InnerProduct(E_th(uex_h), E_th(v))*ds
l += InnerProduct(Ps*uex_h.Trace(), Ps*v.Trace())*ds
l += beta*h**(-2.0)*InnerProduct(Qs*uex_h.Trace(), Qs*v.Trace())*ds
l += InnerProduct(grad(pex_h).Trace(), v.Trace())*ds
l += InnerProduct(grad(q).Trace(), uex_h.Trace())*ds
l += (pex_h*mu)*ds

l.Assemble()

In [ ]:
# Solve problem and write out solution
up_h =  GridFunction(W)
Ainv = Astar.mat.Inverse(freedofs=W.FreeDofs())
up_h.vec.data = Ainv*l.vec

u_h, p_h, _ = up_h.components

Draw(uex_h)
Draw(u_h)
Draw(pex_h)
Draw(p_h)
vtkfile = VTKOutput(mesh, coefs=[u_h, uex_h, p_h, pex_h], names=["u_h", "uex_h", "p_h", "pex_h"], filename="results_stokes",subdivision=3)
vtkfile.Do(vb=BND)
